# КР по модулю 1 — практическая часть



## Задание 1. (1 балл)

Используйте корпус `lenta.txt` из семинара 2 (ячейка ниже скачает его).

In [2]:
import os
import urllib.request
import zipfile

os.makedirs("data", exist_ok=True)

ZIP_URL = "https://github.com/mannefedov/compling_nlp_hse_course/raw/master/data/lenta.txt.zip"
ZIP_PATH = "data/lenta.txt.zip"
TXT_PATH = "data/lenta.txt"

if not os.path.exists(TXT_PATH):
    print("Скачиваю", ZIP_URL)
    urllib.request.urlretrieve(ZIP_URL, ZIP_PATH)
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall("data")
    print("Готово.")

text = open(TXT_PATH, encoding="utf-8").read()
print("Длина текста:", len(text))
print("Первые 300 символов:", text[:300])

Скачиваю https://github.com/mannefedov/compling_nlp_hse_course/raw/master/data/lenta.txt.zip
Готово.
Длина текста: 11536552
Первые 300 символов: Бои у Сопоцкина и Друскеник закончились отступлением германцев. Неприятель, приблизившись с севера к Осовцу начал артиллерийскую борьбу с крепостью. В артиллерийском бою принимают участие тяжелые калибры. С раннего утра 14 сентября огонь достиг значительного напряжения. Попытка германской пехоты про


Одним регулярным выражением найдите в **исходном тексте** корпуса (без какой-либо предобработки) все упоминания дат вида «14 сентября»:
- число от 1 до 31 без ведущего нуля (`5`, `14`, `31`, но не `05` и не `32`);
- затем один или несколько пробельных символов;
- затем название месяца в родительном падеже строчными буквами (`января`, …, `декабря`).

Число и месяц должны быть отдельными словами: из `2014 сентября` не должно извлекаться `14 сентября`, а `5 сентябрям` (если бы такое встретилось) не подходит.

Посчитайте, сколько раз повторяется каждая дата. Найдите топ-2 даты, вставьте в форму вторую по популярности дату (в таком виде - `25 октября`)

In [5]:
import re
from collections import Counter

# Месяцы в родительном падеже
MONTHS = (
    "января|февраля|марта|апреля|мая|июня|июля|августа|"
    "сентября|октября|ноября|декабря"
)


pattern = re.compile(
    rf"(?<!\d)(?<!\w)(3[01]|[12]\d|0?[1-9])(?!\d)\s+({MONTHS})(?!\w)",
    flags=re.UNICODE,
)

matches = pattern.findall(text)
print("Всего совпадений:", len(matches))
print("Первые 10:", matches[:10])

# Собираем даты в нормализованном виде: число + месяц
dates = [f"{int(num)} {month}" for num, month in matches]

counter = Counter(dates)
top = counter.most_common()
print("\nТоп-10 дат:")
for d, c in top[:10]:
    print(f"  {d}: {c}")

# Вторая по популярности - вставляем в форму
if len(top) >= 2:
    second_date = top[1][0]
    print("\nВторая по популярности дата:", second_date)
else:
    print("\nМеньше двух уникальных дат, проверьте данные.")

Всего совпадений: 5133
Первые 10: [('14', 'сентября'), ('16', 'сентября'), ('16', 'сентября'), ('20', 'сентября'), ('31', 'августа'), ('1', 'сентября'), ('1', 'августа'), ('17', 'августа'), ('30', 'августа'), ('30', 'августа')]

Топ-10 дат:
  1 января: 131
  19 декабря: 102
  26 марта: 82
  1 июля: 68
  31 декабря: 64
  1 февраля: 63
  1 октября: 55
  1 апреля: 47
  1 ноября: 42
  1 декабря: 41

Вторая по популярности дата: 19 декабря


### Задание 2 (1 балл)

Найдите самое вероятное продолжение для биграммы `запрет на` в корпусе `data/lenta.txt`

Для этого приведите весь текст к нижнему регистру и сделайте простую токенизацию - разбейте все по пробелам, удалите пунктуацию по границам (strip) и отфильтруйте пустые строки, оставшиеся после этих шагов. Далее рассчитайте биграммные и триграммные частоты и на их основе вычислите самое вероятное продолжение данной биграммы. Вставьте самое вероятное следующее слово в форму

In [ ]:
import re
from collections import Counter

# Нижний регистр
low = text.lower()

# Простая токенизация: разбить по пробелам
tokens = low.split()

# Удалить пунктуацию по границам и отфильтровать пустые strip() по умолчанию снимает whitespace, но пунктуацию нет, поэтому явно снимаем пунктуацию с обоих концов.
def strip_punct(tok: str) -> str:
    return tok.strip('.,!?;:"\'«»()[]{}—–-…')

tokens = [strip_punct(t) for t in tokens]
tokens = [t for t in tokens if t]  # отбрасываем пустые

print("Всего токенов:", len(tokens))
print("Первые 20 токенов:", tokens[:20])

# Биграммы и триграммы
bigrams = Counter(zip(tokens, tokens[1:]))
trigrams = Counter(zip(tokens, tokens[1:], tokens[2:]))

# Продолжение биграммы «запрет на»
target = ("запрет", "на")
print("\nЧастота биграммы 'запрет на':", bigrams.get(target, 0))

# Собираем все триграммы, начинающиеся с "запрет на"
continuations = Counter()
for (w1, w2, w3), c in trigrams.items():
    if w1 == "запрет" and w2 == "на":
        continuations[w3] += c

print("\nТоп-10 продолжений:")
for w, c in continuations.most_common(10):
    print(f"  {w}: {c}")

# Самое вероятное продолжение (MLE по триграммам)
if continuations:
    best_word = continuations.most_common(1)[0][0]
    print("\nСамое вероятное продолжение:", best_word)
else:
    print("\nПродолжений не найдено")

Всего токенов: 1492507
Первые 20 токенов: ['бои', 'у', 'сопоцкина', 'и', 'друскеник', 'закончились', 'отступлением', 'германцев', 'неприятель', 'приблизившись', 'с', 'севера', 'к', 'осовцу', 'начал', 'артиллерийскую', 'борьбу', 'с', 'крепостью', 'в']

Частота биграммы 'запрет на': 84

Топ-10 продолжений:
  публикацию: 9
  запуски: 5
  распространение: 4
  ввоз: 4
  торговлю: 4
  трансляцию: 3
  импорт: 3
  экспорт: 2
  проведение: 2
  воздушное: 2

Самое вероятное продолжение: публикацию


### Задание 3 (1 балл)

Загрузите датасет новостей `lenta_40k` из семинара 3 (ячейка ниже). Сделайте подвыборку из 500 текстов: `lenta.sample(500, random_state=42)` (так у всех получится одна и та же выборка)

Посчитайте эмбеддинги этих 500 текстов моделью `intfloat/multilingual-e5-base`, добавив к каждому тексту префикс `passage: `. 
Посчитайте эмбеддинг текста `Во Франции сотни тысяч людей снова вышли на протесты, полиция применила слезоточивый газ` (с префиксом `query: `). Найдите самый близкий к этому тексту документ по косинусной близости. Вставьте первое слово этого документа в форму как ответ (регистр не важен).

In [11]:
import pandas as pd

lenta = pd.read_csv(
    "https://github.com/mannefedov/compling_nlp_hse_course/raw/master/data/lenta_40k.csv.zip"
).sample(500, random_state=42)

print("Размер выборки:", lenta.shape)
print("Колонки:", lenta.columns.tolist())
print(lenta.head(3))

import os
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel

MODEL_NAME = "intfloat/multilingual-e5-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).eval().to("cpu")
torch.set_num_threads(4)


def mean_pooling(last_hidden_state, attention_mask):
    """Mean pooling с учётом attention_mask"""
    mask = attention_mask.unsqueeze(-1).float()
    return (last_hidden_state * mask).sum(1) / mask.sum(1).clamp(min=1e-9)


@torch.no_grad()
def embed(texts, prefix, batch_size=16):
    """Возвращает L2-нормированные эмбеддинги с префиксом"""
    all_embs = []
    for i in range(0, len(texts), batch_size):
        batch = [prefix + t for t in texts[i:i + batch_size]]
        enc = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt",
        ).to(model.device)
        out = model(**enc)
        emb = mean_pooling(out.last_hidden_state, enc["attention_mask"])
        emb = torch.nn.functional.normalize(emb, p=2, dim=1)
        all_embs.append(emb.cpu().numpy())
    return np.vstack(all_embs)


doc_texts = lenta["text"].astype(str).tolist()
doc_embs = embed(doc_texts, "passage: ")
print("Эмбеддинги документов:", doc_embs.shape)

query_text = "Во Франции сотни тысяч людей снова вышли на протесты, полиция применила слезоточивый газ"
query_emb = embed([query_text], "query: ")
print("Эмбеддинг запроса:", query_emb.shape)

sims = (doc_embs @ query_emb.T).squeeze()
best_idx = int(np.argmax(sims))
best_text = doc_texts[best_idx]
best_score = float(sims[best_idx])
first_word = best_text.split()[0] if best_text.split() else ""

print(f"\nЛучший документ (idx={best_idx}, score={best_score:.4f}):")
print(best_text[:500])

Размер выборки: (500, 2)
Колонки: ['text', 'topic']
                                                    text        topic
12038  Спикер Верховной Рады, исполняющий обязанности...  Бывший СССР
11099  Член комитета Госдумы по уголовному и процессу...       Россия
33791  Минюст России вынес предупреждения 12 некоммер...       Россия


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 396.91it/s]


Эмбеддинги документов: (500, 768)
Эмбеддинг запроса: (1, 768)

Лучший документ (idx=195, score=0.8326):
Беспорядки в пригородах Парижа, населенных преимущественно иммигрантами из Северной Африки, не стихают уже шестой день подряд. Как сообщает Associated Press со ссылкой на местные власти, столкновения полиции с группами молодежи проходят, по меньшей мере, в девяти городах. Беспорядками охвачен департамент Сена-Сен-Дени, расположенный к северу и северо-востоку от центра Парижа. Количество сожженных автомобилей приближается к 70. Всего в регионе зафиксированы около 150 поджогов. Молодежь забрасывае
